# Probar el modo empresa en Kaggle

Este notebook corre la empresa (25 agentes + orquestador LangGraph) en la máquina virtual gratuita de Kaggle. No necesitás computadora propia: solo el celular para subir este archivo a Kaggle.

**Antes de correr, una sola vez:**

1. Subí este archivo a Kaggle: *Create → New Notebook → Upload* (o pegá las celdas en un notebook nuevo).
2. En el notebook: *Add-ons → Secrets* → agregá dos secretos:
   - `GITHUB_TOKEN`: token de GitHub con acceso al repo privado `company-os` (en github.com: *Settings → Developer settings → Personal access tokens → Generate new token (classic)*, tildá `repo`).
   - `LLM_API_KEY`: tu API key del proveedor de LLM.
3. Verificá que el notebook tenga internet: *Settings → Internet → ON*.
4. Editá la **celda 5** con tu proveedor y los modelos que quieras usar.

Después corrés las celdas en orden. La prueba rápida (celda 7) gasta centavos; el modo empresa (celda 9) hace ~21 llamadas al modelo.

In [ ]:
# Celda 2: dependencias
!pip install -q openai pyyaml python-dotenv langgraph

In [ ]:
# Celda 3: secretos (Add-ons → Secrets en Kaggle)
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
GITHUB_TOKEN = secrets.get_secret("GITHUB_TOKEN")
LLM_API_KEY = secrets.get_secret("LLM_API_KEY")
assert GITHUB_TOKEN, "Falta el secreto GITHUB_TOKEN (Add-ons → Secrets)"
assert LLM_API_KEY, "Falta el secreto LLM_API_KEY (Add-ons → Secrets)"
print("Secretos cargados OK (los valores no se muestran).")

In [ ]:
# Celda 4: clonar el repo privado (el token no queda guardado en el repo clonado)
import subprocess, base64, os

if not os.path.exists("company-os"):
    auth = base64.b64encode(f"x-access-token:{GITHUB_TOKEN}".encode()).decode()
    subprocess.run(
        ["git", "-c", f"http.extraHeader=AUTHORIZATION: basic {auth}",
         "clone", "--depth", "1", "https://github.com/enrrutador/company-os.git"],
        check=True, capture_output=True)
    print("Repo clonado.")
else:
    print("El repo ya estaba clonado.")

In [ ]:
# Celda 5: ═══ EDITÁ ESTO SEGÚN TU PROVEEDOR ═══
import os

# Endpoint compatible con la API de OpenAI. Ejemplos:
#   OpenAI:     https://api.openai.com/v1
#   OpenRouter: https://openrouter.ai/api/v1
os.environ["LITELLM_BASE_URL"] = "https://api.openai.com/v1"
os.environ["LITELLM_MASTER_KEY"] = LLM_API_KEY

# Modelos reales para cada nivel (nombres de TU proveedor, no los lógicos empresa-*)
os.environ["MODELO_EMPRESA_BASE"] = "gpt-4o-mini"
os.environ["MODELO_EMPRESA_RAZONAMIENTO"] = "gpt-4o-mini"  # para probar barato; en serio usá uno más capaz
os.environ["MODELO_EMPRESA_LIGERO"] = "gpt-4o-mini"

print("Config OK:", os.environ["LITELLM_BASE_URL"])
print("Modelos:", os.environ["MODELO_EMPRESA_BASE"], "/", os.environ["MODELO_EMPRESA_RAZONAMIENTO"])

## Prueba rápida: un solo agente

Si esto responde, todo el cableado funciona (repo → prompt → tu proveedor → auditoría).

In [ ]:
# Celda 7: smoke test
!cd company-os/runtime && python3 ejecutar.py conciliador "Confirmá en una línea que estás corriendo en Kaggle." 2>&1 | tail -6

## Modo empresa: pipeline punta a punta

El Gerente General orquesta tesis → validación → construcción → comercial con tu objetivo en lenguaje natural. Tarda unos minutos (~21 llamadas). El despliegue frena solo: queda `PENDIENTE_APROBACION_FABIAN` hasta que vos lo apruebes.

In [ ]:
# Celda 9: ═══ EDITÁ TU OBJETIVO ═══
import os
os.environ["OBJETIVO"] = "Creá una web que compare precios en tiempo real de los hipermercados argentinos"
print("Objetivo:", os.environ["OBJETIVO"])

In [ ]:
# Celda 10: correr el modo empresa
!cd company-os/runtime && python3 ejecutar_pipeline.py producto --objetivo "$OBJETIVO" 2>&1 | tail -45

## Ver la auditoría

Cada ejecución de cada agente queda registrada con modelo y tokens.

In [ ]:
# Celda 12: auditoría
!cd company-os/runtime && python3 -c "
import json
lineas = open('registro/auditoria.jsonl', encoding='utf-8').read().strip().split(chr(10))
print(f'{len(lineas)} ejecuciones auditadas')
for l in lineas[-10:]:
    d = json.loads(l)
    print('-', d['agente'], '|', d['modelo'])
" 2>&1 | tail -14